# Comparação de Segmentação: U-Net vs YOLO26-seg vs SAM 3

In [4]:
import os
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Imports do Keras para a U-Net
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, Conv2DTranspose, concatenate, Dropout
from tensorflow.keras.models import Model

# Imports da Ultralytics para YOLO e SAM
from ultralytics import YOLO, SAM

# ================= CONFIGURAÇÃO DE CAMINHOS =================
BASE_DIR = Path("/home/antoniovinicius/projects")

# Caminhos dos Pesos (Ajustados com base nos melhores modelos registrados)
UNET_WEIGHTS = BASE_DIR / "sandbox_unet/logs/pipeline_unet_v4/phase3_optimized/unet_optimized/weights/best_model.h5"
YOLO_WEIGHTS = BASE_DIR / "sandbox_yolo26/logs/train_isic_2018_task_1_v8/yolo26_small_ft_isic_2018_v8/weights/best.pt"
SAM_WEIGHTS  = BASE_DIR / "sandbox_sam3/logs/pipeline_sam3_v1/phase1_baseline/weights/best.pt"

# Caminhos dos Dados de Validação em Numpy (conforme projeto U-Net)
VAL_IMG_PATH = BASE_DIR / "sandbox_unet/datasets/isic_2018_task1_numpy/ISIC2018_Task1-2_Validation_Input/ISIC2018_Task1-2_Validation_Input.npy"
VAL_GT_PATH  = BASE_DIR / "sandbox_unet/datasets/isic_2018_task1_numpy/ISIC2018_Task1_Validation_GroundTruth/ISIC2018_Task1_Validation_GroundTruth.npy"

# Índice da imagem a ser avaliada
IMG_INDEX = 53

I0000 00:00:1790596544.335263 2919072 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790596544.452587 2919072 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790596546.354363 2919072 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 1. Carregamento dos Dados
Lendo as matrizes Numpy contendo as imagens e as máscaras Ground Truth de validação e isolando a imagem selecionada.

In [5]:
# Carregamento do dataset de validação
X_val = np.load(VAL_IMG_PATH)
y_val = np.load(VAL_GT_PATH)

print(f"Dados de validação carregados! Imagens: {X_val.shape}, Máscaras: {y_val.shape}")

# Seleciona a imagem e a máscara GT
img_array = X_val[IMG_INDEX]
gt_mask = y_val[IMG_INDEX]

# A U-Net consome a imagem diretamente como float (geralmente normalizado),
# mas os modelos da Ultralytics esperam formato de imagem uint8 [0-255].
if img_array.max() <= 1.0:
    img_uint8 = (img_array * 255).astype(np.uint8)
else:
    img_uint8 = img_array.astype(np.uint8)

# Exibe rapidamente as formas e dados extraídos
print(f"Imagem de teste isolada com sucesso. Shape U-Net: {img_array.shape}, Shape YOLO/SAM: {img_uint8.shape}")

Dados de validação carregados! Imagens: (100, 256, 256, 3), Máscaras: (100, 256, 256, 1)
Imagem de teste isolada com sucesso. Shape U-Net: (256, 256, 3), Shape YOLO/SAM: (256, 256, 3)


## 2. Inferência com U-Net (Otimizada)
Recriando a arquitetura com os hiperparâmetros da fase de HPO (`n_filters=32`, `dropout=0.2959`) e carregando os pesos `.h5`.

In [6]:
def conv2d_block(input_tensor, n_filters, kernel_size=3, batchnorm=True):
    x = Conv2D(filters=n_filters, kernel_size=(kernel_size, kernel_size), kernel_initializer="he_normal", padding="same")(input_tensor)
    if batchnorm: x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(filters=n_filters, kernel_size=(kernel_size, kernel_size), kernel_initializer="he_normal", padding="same")(x)
    if batchnorm: x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

def get_unet(input_img, n_filters, dropout, batchnorm):
    c1 = conv2d_block(input_img, n_filters=n_filters*1, batchnorm=batchnorm)
    p1 = Dropout(dropout)(MaxPooling2D((2, 2))(c1))
    c2 = conv2d_block(p1, n_filters=n_filters*2, batchnorm=batchnorm)
    p2 = Dropout(dropout)(MaxPooling2D((2, 2))(c2))
    c3 = conv2d_block(p2, n_filters=n_filters*4, batchnorm=batchnorm)
    p3 = Dropout(dropout)(MaxPooling2D((2, 2))(c3))
    c4 = conv2d_block(p3, n_filters=n_filters*8, batchnorm=batchnorm)
    p4 = Dropout(dropout)(MaxPooling2D(pool_size=(2, 2))(c4))
    c5 = conv2d_block(p4, n_filters=n_filters*16, batchnorm=batchnorm)
    u6 = Conv2DTranspose(n_filters*8, (3, 3), strides=(2, 2), padding='same')(c5)
    c6 = conv2d_block(Dropout(dropout)(concatenate([u6, c4])), n_filters=n_filters*8, batchnorm=batchnorm)
    u7 = Conv2DTranspose(n_filters*4, (3, 3), strides=(2, 2), padding='same')(c6)
    c7 = conv2d_block(Dropout(dropout)(concatenate([u7, c3])), n_filters=n_filters*4, batchnorm=batchnorm)
    u8 = Conv2DTranspose(n_filters*2, (3, 3), strides=(2, 2), padding='same')(c7)
    c8 = conv2d_block(Dropout(dropout)(concatenate([u8, c2])), n_filters=n_filters*2, batchnorm=batchnorm)
    u9 = Conv2DTranspose(n_filters*1, (3, 3), strides=(2, 2), padding='same')(c8)
    c9 = conv2d_block(Dropout(dropout)(concatenate([u9, c1], axis=3)), n_filters=n_filters*1, batchnorm=batchnorm)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c9)
    return Model(inputs=[input_img], outputs=[outputs])

# Instanciar e carregar a U-Net
print(f"Carregando pesos U-Net de: {UNET_WEIGHTS.name}")
input_img = Input((256, 256, 3))
model_unet = get_unet(input_img, n_filters=32, dropout=0.2959, batchnorm=True)
model_unet.load_weights(str(UNET_WEIGHTS))

# Predição
unet_pred_raw = model_unet.predict(np.expand_dims(img_array, axis=0))[0]
unet_mask = (unet_pred_raw > 0.5).astype(np.uint8)
print("Inferência da U-Net concluída.")

Carregando pesos U-Net de: best_model.h5


I0000 00:00:1790596568.325600 2919072 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 505 MB memory:  -> device: 0, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:17:00.0, compute capability: 7.0
I0000 00:00:1790596568.326418 2919072 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13042 MB memory:  -> device: 1, name: Tesla V100S-PCIE-32GB, pci bus id: 0000:65:00.0, compute capability: 7.0
I0000 00:00:1790596571.046018 2923305 service.cc:153] XLA service 0x72556c0405a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790596571.046053 2923305 service.cc:161]   StreamExecutor [0]: Tesla V100S-PCIE-32GB, Compute Capability 7.0 (Driver: 13.0.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1790596571.046067 2923305 service.cc:161]   StreamExecutor [1]: Tesla V100S-PCIE-32GB, Compute Capability 7.0 (Driver: 13.0.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9

1/1 ━━━━━━━━━━━━━━━━━━━━ 178s 178s/step
Inferência da U-Net concluída.


I0000 00:00:1790596748.450211 2923305 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


## 3. Inferência com YOLO26-seg (Champion v8 Small)

In [7]:
print(f"Carregando modelo YOLO26-seg de: {YOLO_WEIGHTS.name}")
model_yolo = YOLO(str(YOLO_WEIGHTS))

# Inferência com verbose=False para manter o notebook limpo
yolo_res = model_yolo.predict(img_uint8, imgsz=256, verbose=False)[0]

# Extraindo a máscara 2D gerada pelo YOLO
if yolo_res.masks is not None:
    yolo_mask = yolo_res.masks.data[0].cpu().numpy()
else:
    print("Nenhuma máscara detectada pelo YOLO26!")
    yolo_mask = np.zeros((256, 256))

# A máscara do YOLO às vezes precisa de redimensionamento caso o input layer tenha mudado
if yolo_mask.shape != (256, 256):
    yolo_mask = cv2.resize(yolo_mask, (256, 256), interpolation=cv2.INTER_NEAREST)

print("Inferência YOLO26-seg concluída.")

Carregando modelo YOLO26-seg de: best.pt
Nenhuma máscara detectada pelo YOLO26!
Inferência YOLO26-seg concluída.


## 4. Inferência com SAM 3 (Baseline Phase 1)

In [8]:
# Nota: Dependendo de como a biblioteca ultralytics lidou com o SAM, a chamada pode usar o SAM(..) diretamente.
print(f"Carregando modelo SAM 3 de: {SAM_WEIGHTS.name}")

try:
    model_sam = SAM(str(SAM_WEIGHTS))
except Exception as e:
    print(f"Fallback para classe YOLO caso seja um fine-tuning via YOLO class. Erro: {e}")
    model_sam = YOLO(str(SAM_WEIGHTS))

sam_res = model_sam.predict(img_uint8, imgsz=256, verbose=False)[0]

if sam_res.masks is not None:
    sam_mask = sam_res.masks.data[0].cpu().numpy()
else:
    print("Nenhuma máscara detectada pelo SAM 3!")
    sam_mask = np.zeros((256, 256))

if sam_mask.shape != (256, 256):
    sam_mask = cv2.resize(sam_mask, (256, 256), interpolation=cv2.INTER_NEAREST)

print("Inferência SAM 3 concluída.")

Carregando modelo SAM 3 de: best.pt
Fallback para classe YOLO caso seja um fine-tuning via YOLO class. Erro: /home/antoniovinicius/projects/sandbox_sam3/logs/pipeline_sam3_v1/phase1_baseline/weights/best.pt is not a supported SAM model. Available models are: 
 dict_keys(['sam_h.pt', 'sam_l.pt', 'sam_b.pt', 'mobile_sam.pt', 'sam2_t.pt', 'sam2_s.pt', 'sam2_b.pt', 'sam2_l.pt', 'sam2.1_t.pt', 'sam2.1_s.pt', 'sam2.1_b.pt', 'sam2.1_l.pt'])


FileNotFoundError: [Errno 2] No such file or directory: '/home/antoniovinicius/projects/sandbox_sam3/logs/pipeline_sam3_v1/phase1_baseline/weights/best.pt'

## 5. Visualização Qualitativa
Plotando a imagem original, a máscara Ground Truth (GT) e as predições dos 3 modelos lado a lado.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 6))
plt.style.use("seaborn-v0_8-whitegrid")

# 1. Imagem Original
axes[0].imshow(img_uint8)
axes[0].set_title(f"Original (idx: {IMG_INDEX})", fontsize=14, fontweight="bold")
axes[0].axis('off')

# 2. Ground Truth
axes[1].imshow(gt_mask.squeeze(), cmap='gray')
axes[1].set_title("Ground Truth", fontsize=14, fontweight="bold")
axes[1].axis('off')

# 3. U-Net Predict
axes[2].imshow(unet_mask.squeeze(), cmap='magma')
axes[2].set_title("U-Net (HPO v4)", fontsize=14, fontweight="bold")
axes[2].axis('off')

# 4. YOLO26 Predict
axes[3].imshow(yolo_mask, cmap='viridis')
axes[3].set_title("YOLO26-seg (v8 Small)", fontsize=14, fontweight="bold")
axes[3].axis('off')

# 5. SAM 3 Predict
axes[4].imshow(sam_mask, cmap='plasma')
axes[4].set_title("SAM 3 (v1 Baseline)", fontsize=14, fontweight="bold")
axes[4].axis('off')

plt.tight_layout()
plt.show()